# Trabajo Fin de Máster  
### Análisis de la Ciudad mediante Aprendizaje Supervisado  
#### Detección Automática de Tipologías Residenciales y Patrones de Cerramiento: Interpretabilidad vs Rendimiento

**Master Universitario en Ciencia de Datos e Ingeniería de Computadores (Universidad de Granada)**

> **Autor:** David Fernández Martínez    
> **Email personal:** david.fernxndez.martinez@gmail.com  
> **Email académico:** davidfm8@correo.ugr.es  
> **LinkedIn:** [linkedin.com/in/david-fernández-martínez](https://www.linkedin.com/in/david-fern%C3%A1ndez-mart%C3%ADnez/)  
> **GitHub:** [github.com/davidfernxndez](https://github.com/davidfernxndez)

---

## Diseño experimental de la evaluación de rendimiento

### 📝 Descripción del notebook
En este notebook se presenta el diseño experimental realizado para evaluar el rendimiento de modelos interpretables y "caja negra" sobre el conjunto de datos de complejos residenciales del área metropolitana de Granada (situado en */data/processed/model_data.csv).

Para ello se define la estrategia de validación, las métricas de evaluación y los modelos seleccionados. Finalmente se presenta la implementación experimental desarrollada para realizar el experimento de forma estructurada y reproducible. El análisis de los resultados obtenidos se realiza en el *notebook 04_Performance_results_analysis.ipynb*.

### Indice de contenidos
1. [Contexto y objetivos del experimento](#objetivos)
 
2. [Estrategia de validación: *Nested Cross-Validation*](#nested_cross_validation)

3. [Métricas de evaluación de rendimiento](#metricas_evaluacion)
    * [3.1 Métricas de rendimiento global](#metricas_globales)
        * [3.1.1 F1-score macro](#f1_macro)
        * [3.1.2 Coeficiente de Correlación de Matthews (MCC)](#mcc)
        * [3.1.3 Accuracy](#accuracy)    

    * [3.2 Métricas de rendimiento por clase](#metricas_por_clase)

4. [Modelos interpretables](#modelos_interpretables)
    * [4.1 Regresión logística Multinomial (*Softmax Regression*)](#regresion_logistica)
        * [4.1.1 Fundamento teórico](#regresion_logistica_teoria)
        * [4.1.2 Interpretabilidad del modelo](#regresion_logistica_interpretabilidad)
        * [4.1.3 Implementación y optimización de hiperparámetros.](#regresion_logistica_implementacion)
    * [4.2 Árbol de decisión](#arbol_decision)
        * [4.2.1 Fundamento teórico](#arbol_decision_teoria)
        * [4.2.2 Interpretabilidad del modelo](#arbol_decision_interpretabilidad)
        * [4.2.3 Implementación y optimización de hiperparámetros.](#arbol_decision_implementacion)

5. [Modelos no interpretables ("caja negra")](#modelos_no_interpretables)
    * [5.1 Máquina de vector soporte (SVM) con kernel RBF ](#svm)
        * [5.1.1 Fundamento teórico](#svm_teoria)
        * [5.1.2 Limitaciones de interpretabilidad](#svm_interpretabilidad)
        * [5.1.3 Implementación y optimización de hiperparámetros.](#svm_implementacion)
    * [5.2 Random Forest](#random_forest)
        * [5.2.1 Fundamento teórico](#random_forest_teoria)
        * [5.2.2 Limitaciones de interpretabilidad](#random_forest_interpretabilidad)
        * [5.2.3 Implementación y optimización de hiperparámetros.](#random_forest_implementacion)
    * [5.3 XGBoost (Extreme Gradient Boosting)](#xgboost)
        * [5.3.1 Fundamento teórico](#xgboost_teoria)
        * [5.3.2 Limitaciones de interpretabilidad](#xgboost_interpretabilidad)
        * [5.3.3 Implementación y optimización de hiperparámetros.](#xgboost_implementacion)

6. [Implementación experimental](#implementacion_experimental)

# Configuración de entorno e *imports*

Este proyecto ha sido realizado en un entorno Anaconda con la versión 3.11.15 de *Python*. Las versiones de las librerias requeridas se encuentran en el fichero *requirements.txt*.

En esta sección se importan las librerias necesarias para la ejecución de este fichero jupyter notebook, se activa el *reload* de módulos externos y se configuran aspectos globales y de reproducibilidad.

In [2]:
# jupyter extensions to automatically reload external modules
%load_ext autoreload
%autoreload 2

In [3]:
import warnings
import random
import numpy as np
import pandas as pd
import mlflow
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb
from xgboost import XGBClassifier

# Configuration object
from src.config import cfg

# Nested Cross Validation methods
from src.performance_utils import performance_experiment, performance_experiment_mlflow

**Import troubleshooting**

If the `src` imports fail when running this notebook in a different environment,
uncomment and execute the following cell:

```python
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
```

In [4]:
# Set MLFLOW variable to True if you want to register model evaluation in MLFLOW format
USE_MLFLOW = False

# Set folder to store MLFlow files
mlflow.set_tracking_uri(f"file:{cfg.ROOT_DIR}/mlruns")

In [5]:
# Global configuration
sns.set_theme(style="ticks", context="notebook")
plt.rcParams["font.family"] = "sans-serif"
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
warnings.filterwarnings("ignore")

In [6]:
# Reproducibility
SEED = cfg.SEED 
np.random.seed(SEED)
random.seed(SEED)

<a id="objetivos"></a>
# 1. Contexto y objetivos del experimento

Esta sección introduce los objetivos del experimento de evaluación del rendimiento realizado sobre el conjunto de datos de complejos residenciales del área metropolitana de Granada (situado en */data/processed/model_data.csv).

La evaluación del rendimiento se enmarca en el análisis del compromiso existente (*trade-off*) entre la capacidad predictiva y la interpretabilidad de los modelos de aprendizaje automático. Con este propósito, se consideran dos tipos de modelos:

* **Modelos interpretables (o "caja blanca")**. Aquellos cuyo funcionamiento interno puede entenderse de forma directa sin necesidad de recurrir a herramientas externas.
* **Modelos no interpretables ("caja negra")**. Algoritmos cuyo proceso interno es demasiado complejo para que una persona pueda comprender de forma sencilla porque generan una determinada predicción. El análisis de interpretabilidad de estos modelos requiere el uso de herramientas *post-hoc* de inteligencia artificial explicable (XAI) que proporcionan técnicas para comprender y justificar las decisiones de modelos complejos.

La selección de modelos realizada persigue cubrir diferentes enfoques de interpretabilidad intrínseca, así como distintos niveles de capacidad predictiva. De este modo, se pretende obtener una visión amplia del compromiso entre interpretabilidad y rendimiento, a medida que la complejidad de los algoritmos crece.

En general, los modelos "caja negra" suelen alcanzar un rendimiento predictivo superior al de los modelos interpretables, debido a su mayor capacidad para capturar relaciones complejas y no lineales entre las variables. Por ello, el principal objetivo de esta experimentación consiste en cuantificar la ganancia de rendimiento obtenida al emplear modelos más complejos frente a modelos interpretables. Esto permite evaluar el coste de oportunidad asociado a priorizar la interpretabilidad del modelo frente al posible incremento en capacidad predictiva.

Los resultados obtenidos, junto con el posterior análisis de la interpretabilidad de los modelos, permitirán determinar cuáles son las estrategias más adecuadas para abordar este problema de clasificación. Este conjunto de datos se enmarca en el análisis del desarrollo urbanístico de la ciudad desde una perspectiva sociológica, por lo que resulta especialmente relevante que los investigadores puedan comprender las razones que justifican la clasificación de un complejo residencial dentro de una determinada tipología de cerramiento.

Además, un buen nivel de interpretabilidad facilita la identificación de las variables urbanísticas con mayor influencia en la clasificación. Esta información puede resultar de gran utilidad para optimizar futuras campañas de recopilación de datos, permitiendo priorizar aquellas variables más relevantes o mejorar la calidad de aquellas variables de interés que no están contribuyendo de forma significativa a la capacidad predictiva del modelo.

En las secciones siguientes se describe el diseño experimental adoptado para la evaluación del rendimiento, detallando la estrategia de validación, las métricas de evaluación y los modelos seleccionados. Finalmente, se presenta la implementación experimental desarrollada en Python para la ejecución de los experimentos.

<a id="nested_cross_validation"></a>
# 2. Estrategia de validación: *Nested Cross-Validation*

La estrategia de validación utilizada en este experimento es la validación cruzada anidada (*Nested Cross Validation*) presentada en el *notebook 02_Nested_Cross_Validation.ipynb*. Este enfoque permite separar completamente la optimización del modelo (selección de hiperparámetros) de la evaluación de su rendimiento, proporcionando una estimación más fiable de la capacidad de generalización del modelo y aproximándose al rendimiento que se obtendría sobre un conjunto de prueba independiente. 

La metodología consiste en ejecutar dos procesos de validación cruzada anidados:
* **Validación cruzada externa (*Outer CV Loop*)**: se utiliza para evaluar el rendimiento del modelo sobre datos que no han intervenido en ninguna fase del ajuste de hiperparámetros. Para ello, se define un total de $K_{out}$ particiones (*folds*).
* **Validación cruzada interna (*Inner CV Loop*)**: se utiliza exclusivamente para la optimización del modelo mediante la selección de hiperparámetros. Este proceso se realiza únicamente sobre los datos de entrenamiento definidos en cada iteración de la validación externa. Para ello, se establecen $K_{in}$ particiones (*folds*) adicionales.

Al finalizar el proceso, se obtienen $K_{out}$ estimaciones independientes del rendimiento del modelo, una por cada iteración de la validación cruzada externa. Estas estimaciones permiten calcular métricas promedio y analizar su variabilidad, proporcionando una evaluación más robusta del comportamiento esperado del modelo sobre datos no observados.

Se ha optado por este enfoque debido a que se trabaja con un conjunto de datos pequeño. En lugar de congelar un $20\%$ o $30\%$ de los datos para evaluación, la totalidad de los datos son utilizados tanto para entrenar como para evaluar en diferentes momentos del proceso. Por lo tanto, el rendimiento promedio está respaldado por todo el *dataset* y no depende de una única partición fija, la cual podría presentar una alta varianza debido a la escasez de datos.

La configuración experimental utilizada, definida en el archivo *src/config.py*, es la siguiente:
* *OUTER_SPLITS $(K_{out})$* = 5.
* *INNER_SPLITS $(K_{in})$* = 3.

Con el objetivo de garantizar reproducibilidad y una comparación justa entre modelos, las particiones empleadas tanto en la validación cruzada externa como en la interna se almacenan previamente en ficheros CSV (ver directorio */data/folds/*). De esta forma, todos los modelos son entrenados y evaluados utilizando exactamente las mismas particiones de datos durante todo el proceso experimental.

<a id="metricas_evaluacion"></a>
# 3. Métricas de evaluación de rendimiento

Para evaluar el rendimiento de los modelos de clasificación, se consideran tanto métricas globales como métricas específicas por clase. Las métricas globales proporcionan una visión agregada del rendimiento del modelo, permitiendo cuantificar su capacidad predictiva global sobre el conjunto de datos. Por su parte, las métricas específicas por clase ofrecen un análisis más detallado, al evaluar el rendimiento alcanzado en cada una de las categorías de la variable objetivo.

La combinación de ambos niveles de análisis permite realizar una evaluación más completa del rendimiento de los modelos. Además de cuantificar las diferencias de rendimiento global entre modelos interpretables y modelos "caja negra2, este enfoque permite identificar en qué clases concretas se producen dichas diferencias. De este modo, es posible determinar no solo si un modelo ofrece un mejor rendimiento global, sino también qué categorías se benefician en mayor medida de la utilización de modelos de mayor complejidad.

<a id="metricas_globales"></a>
## 3.1 Métricas de rendimiento global

La estrategia Nested Cross-Validation proporciona $K_{out}$ estimaciones independientes del rendimiento, una por cada partición del bucle externo. En este experimento se ha utilizado una configuración de $K_{out} = 5$ folds, por lo que cada métrica queda representada por un conjunto de cinco valores.

Esto permite analizar el rendimiento de los modelos desde dos perspectivas complementarias. Por un lado, se evalúa su capacidad predictiva mediante la media y la desviación estándar de las métricas obtenidas en las distintas particiones. Por otro lado, se analiza la estabilidad del modelo a través de la variabilidad del rendimiento entre folds, lo que permite identificar aquellos modelos más robustos frente a cambios en las particiones de entrenamiento y test.

Para la definición formal de las métricas empleadas en el experimento, se introduce en primer lugar la matriz de confusión generalizada, que es la extensión de la matriz de confusión binaria al caso de clasificación multiclase.

Sea un problema de clasificación multiclase con $K$ categorías mutuamente excluyentes, la matriz de confusión generalizada se define como una matriz cuadrada $C \in \mathbb{R}^{K \times K}$, donde cada elemento $C_{ij}$ representa el número de observaciones cuya clase real pertenece a la categoría $i$ (filas de la matriz) y que han sido clasificadas por el modelo como pertenecientes a la categoría $j$ (columnas de la matriz).

$$\mathbf{C} = (C_{ij})_{K \times K} = \begin{pmatrix}
C_{11} & C_{12} & \cdots & C_{1K} \\
C_{21} & C_{22} & \cdots & C_{2K} \\
\vdots & \vdots & \ddots & \vdots \\
C_{K1} & C_{K2} & \cdots & C_{KK}
\end{pmatrix}, \quad \text{donde } C_{ij} \text{ con } i, j = 1, \dots, K$$


A partir de esta matriz, se define para cada clase $k$ (con $k \in {1,\ldots,K}$), las siguientes cantidades:

* Verdaderos positivos (*True Positives, $TP_k$*). Corresponden al número de observaciones de la clase $k$ correctamente clasificadas. Estos valores se encuentran en la diagonal principal de la matriz de confusión:
$$TP_k=C_{kk}$$

* Falsos positivos (*False Positives, $FP_k$*). Representan las observaciones pertenecientes a otras clases que han sido clasificadas erróneamente como la clase $k$ (Error tipo I). Se obtienen sumando los elementos de la columna $k$, excluyendo el elemento diagonal:
$$FP_k = \sum_{i=1, i \neq k}^{K} C_{ik}$$

* Falsos negativos (*False Negatives, $FN_k$*). Representan las observaciones cuya clase real es $k$ y que han sido clasificadas erróneamente en una categoría distinta (Error tipo II). Se obtienen sumando los elementos de la fila $k$, excluyendo el elemento diagonal:
$$FN_k = \sum_{j=1, j \neq k}^{K} C_{kj}$$

<a id="f1_macro"></a>
### 3.1.1 F1-score macro

Para la evaluación y comparación de los modelos de clasificación es necesario considerar el desbalanceo de clases presente en el conjunto de datos. Tal y como se ha observado en el análisis exploratorio de datos, y como se ilustra en la figura siguiente, la variable objetivo presenta un desbalanceo moderado, con dos clases mayoritarias y tres clases minoritarias, siendo la relación entre la clase más frecuente (Controlado) y la menos representada (Autoaislado) de 261:38.

<img src="../images/imbalanced_class.jpg" alt="Esquema de Validación Cruzada Anidada" width="60%">



En este contexto, la métrica estándar de accuracy no resulta adecuada como criterio principal de evaluación, ya que puede verse dominada por el rendimiento en las clases mayoritarias, enmascarando un comportamiento deficiente en las clases minoritarias. Por este motivo, es necesario emplear una métrica que refleje de forma equilibrada el comportamiento del modelo sobre todas las clases.

Se ha seleccionado como métrica principal de evaluación el F1-score macro, implementado en la librería *scikit-learn*. Esta métrica se emplea tanto para la evaluación del rendimiento como para la optimización de hiperparámetros en la validación cruzada interna (*Inner CV*) del esquema Nested Cross-Validation.

El F1-score se define como la media armónica entre la precisión y el *recall* o sensibilidad. En el contexto de este problema urbanístico no existe una preferencia por minimizar falsos positivos frente a falsos negativos, o viceversa, sino que se busca un equilibrio entre la exactitud de la clasificación (precisión) y la capacidad de capturar el mayor número de complejos residenciales reales de cada tipología (*Recall*). Por lo tanto, el F1-score constituye una métrica adecuada para evaluar el rendimiento desde una perspectiva equilibrada entre ambas conceptos.

Para abordar explícitamente el desbalanceo de clases, se emplea el esquema de agregación *macro* disponible en *scikit-learn*, en lugar de las alternativas *micro* o *weighted*. El F1-score macro se obtiene calculando el F1-score de cada clase de forma independiente y promediando posteriormente todos los valores mediante una media aritmética no ponderada. De este modo, todas las clases contribuyen por igual al resultado final, independientemente de su frecuencia en el conjunto de datos. En consecuencia, el rendimiento en las clases minoritarias tiene un impacto equivalente al de las clases mayoritarias, evitando que un buen desempeño en estas últimas oculte posibles deficiencias en las categorías menos representadas.

Para una clase individual $k$, la precisión ($P$) y el recall ($R$) se calculan a partir la matriz de confusión generalizada como:
$$P_k = \frac{TP_k}{TP_k + FP_k} \quad ; \quad R_k = \frac{TP_k}{TP_k + FN_k}$$

El F1-score de la clase $k$ se calcula como la media armónica entre la precisión y el recall:
$$F_{1,k} = 2 \cdot \frac{P_k \cdot R_k}{P_k + R_k}$$

Finalmente el F1-score Macro se obtiene mediante la media aritmética del F1-score de cada una de las $K$ clases:
$$F_1\text{-Score Macro} = \frac{1}{K} \sum_{k=1}^{K} F_{1,k}$$

<a id="mcc"></a>
#### 3.1.2 Coeficiente de Correlación de Matthews (MCC)

El coeficiente de correlación de Matthews (MCC) se incorpora como métrica complementaria al F1-score macro, ya que proporciona una visión alternativa del rendimiento del modelo en presencia de desbalanceo de clases (citar *Chicco, D., & Jurman, G. (2020). The advantages of the Matthews correlation coefficient (MCC) over F1 score and accuracy in binary classification evaluation*).

Mientras que el F1-score macro se calcula como la media no ponderada de los F1-score obtenidos de forma independiente para cada clase, el MCC proporciona una medida global que tiene en cuenta simultáneamente todos los elementos de la matriz de confusión, es decir, verdaderos positivos, verdaderos negativos, falsos positivos y falsos negativos.

Esta métrica evalúa el grado de concordancia entre las etiquetas reales y las predicciones, interpretando ambos conjuntos como variables categóricas cuya correlación se desea cuantificar. En su formulación para problemas multiclase, el MCC se define a partir de la matriz de confusión de la siguiente forma:

$$MCC = \frac{
    c \times s - \sum_{k}^{K} p_k \times t_k
}{\sqrt{
    (s^2 - \sum_{k}^{K} p_k^2) \times
    (s^2 - \sum_{k}^{K} t_k^2)
}}$$

donde:
* $t_k=\sum_{j}^{K} C_{kj}$ representa la frecuencia real de la clase $k$ (suma de la fila $k$ en la matriz de confusión generalizada).
* $p_k=\sum_{i}^{K} C_{ik}$ representa el total de predicciones asignadas a la clase $k$ (suma de la columna $k$ en la matriz de confusión generalizada).
* $c=\sum_{k}^{K} C_{kk}$ es el número de muestras predichas correctamente.
* $s=\sum_{i}^{K} \sum_{j}^{K} C_{ij}$ es el número total de muestras.

El numerador de la métrica cuantifica la divergencia entre el rendimiento observado del clasificador y el rendimiento atribuible al azar. El término $c \cdot s$ pondera los aciertos reales del modelo en relación con el tamaño de la muestra, mientras que el término asociativo $\sum p_k \cdot t_k$ modela matemáticamente la esperanza de aciertos que se obtendrían mediante una asignación puramente aleatoria que respetase las proporciones de las clases.El denominador actúa como un factor de normalización basado en las varianzas de las distribuciones reales y predichas. A diferencia del caso binario, en problemas de clasificación multiclase este factor acota el índice en un rango donde el límite superior es siempre $+1$ (clasificación perfecta) y el valor $0$ representa un desempeño equivalente al azar, mientras que el límite inferior negativo oscila entre $0$ y $-1$ en función del grado de desbalanceo y el número de categorías.

Esta métrica se ha implementado mediante la función *matthews_corrcoef()* de la librería *scikit-learn*.

<a id="accuracy"></a>
### 3.1.3 Accuracy

El *accuracy* se incluye en el conjunto de métricas globales al tratarse de la medida de rendimiento más ampliamente utilizada en problemas de clasificación, lo que facilita la comparabilidad con trabajos futuros sobre este conjunto de datos. No obstante, debido al desbalanceo de clases, la interpretación del rendimiento y comparación entre modelos se realiza a partir de las métricas F1-Score Macro y MCC.

El *accuracy* representa la proporción total de aciertos sobre el total de la muestra ($N$). Se calcula sumando la diagonal de la matriz de confusión generalizada y dividiendo entre la suma de todos sus elementos:

$$\text{Accuracy} = \frac{\sum_{k=1}^{K} C_{kk}}{\sum_{i=1}^{K} \sum_{j=1}^{K} C_{ij}} = \frac{\sum_{k=1}^{K} TP_k}{N}$$

Esta métrica se ha a implementado mediante la funcion *accuracy_score()* del paquete *metrics* de la librería *scikit-learn*.

<a id="metricas_por_clase"></a>
## 3.2 Métricas de rendimiento por clase

Para obtener una evaluación detallada y desagregada del rendimiento de los clasificadores en cada una de las cinco tipologías de cerramiento, las métricas por clase no se calculan a partir del promedio de los resultados obtenidos en cada fold, sino a partir de la agregación de las predicciones de la validación cruzada externa.

En el esquema Nested Cross Validation, los conjuntos de test (*Outer Test Set*) correspondientes a cada outer fold son mutuamente excluyentes. En consecuencia, al finalizar el experimento se dispone de las predicciones *out-of-fold* para los 642 complejos residenciales del conjunto de datos, donde cada observación ha sido evaluada una sola vez por el modelo óptimo entrenado en su correspondiente partición. 

A partir de estas predicciones y de las etiquetas reales se construye la matriz de confusión generalizada out-of-fold, obtenida mediante la combinación de las predicciones de todas las particiones de la validación cruzada externa. Esta matriz constituye la base para el análisis desagregado del rendimiento del modelo. A partir de ella se calculan las métricas específicas por clase (precisión, recall y F1-score) siguiendo la formulación definida en la sección [3.1.1 F1-Score Macro](#f1_macro).

El cálculo de estas métricas se ha realizado mediante la funcion *classification_report()* del módulo *metrics* de la librería *scikit-learn*.

Este análisis permite identificar las clases más difíciles de clasificar, así como realizar una comparación más detallada entre modelos interpretables y modelos "caja negra". Asimismo, la matriz de confusión *out-of-fold* facilita el análisis de patrones de confusión entre clases, permitiendo identificar aquellas categorías que el modelo tiende a confundir con mayor frecuencia.

<a id="modelos_interpretables"></a>
# 4 Modelos interpretables 

El objetivo de los modelos interpretables es permitir que el experto en el dominio pueda comprender de forma directa el comportamiento aprendido por el modelo, sin necesidad de recurrir a técnicas adicionales de interpretación post-hoc. En este trabajo se han seleccionado dos modelos que incorporan mecanismos de interpretabilidad intrínseca basados en enfoques distintos.

Por un lado, la regresión logística multinomial permite cuantificar la contribución de cada variable predictora a la probabilidad de pertenencia a cada clase, a través de sus coeficientes estimados. Esto proporciona una interpretación directa del efecto marginal de las variables sobre la salida del modelo. Por otro lado, el árbol de decisión ofrece una interpretabilidad basada en reglas lógicas secuenciales del tipo if-then, lo que facilita su comprensión al aproximarse al proceso de toma de decisiones humano mediante particiones sucesivas del espacio de variables.

En términos de interpretabilidad, el árbol de decisión se considera el modelo de referencia, ya que su estructura basada en reglas permite una comprensión directa e intuitiva por parte de expertos en el dominio del problema, sin necesidad de conocimientos técnicos avanzados.

<a id="regresion_logistica"></a>
## 4.1 Regresión logística Multinomial (*Softmax Regression*)

<a id="regresion_logistica_teoria"></a>
### 4.1.1 Fundamento teórico

La regresión logística multinomial, también conocida como regresión Softmax, es una extensión de la regresión logística binaria diseñada para abordar problemas de clasificación donde la variable dependiente $Y$ es de tipo categórica y cuenta con $K$ niveles mutuamente excluyentes. 

El algoritmo modela la probabilidad condicional de que una observación pertenezca a cada una de las categorías dadas las variables predictoras. Para ello, el modelo calcula de forma simultánea una combinación lineal de las variables independientes para cada una de las $K$ clases, generando una puntuación o score no normalizado para cada una de ellas. Posteriormente, estas funciones lineales se transforman a través de la función Softmax para garantizar que las predicciones finales sean probabilidades normalizadas cuya sumatoria sea igual a la unidad (citar *Hosmer Jr, D. W., Lemeshow, S., & Sturdivant, R. X. (2013). Applied logistic regression. John Wiley & Sons*.).

La probabilidad de que una observación representada por el vector de predictores $\mathbf{x}$ pertenezca a la categoría $k$ viene dada por:

$$P(Y = k \mid X) = \frac{e^{\beta_{k0} + \mathbf{\beta}_k^T \mathbf{X}}}{\sum_{j=1}^{K} e^{\beta_{j0} + \mathbf{\beta}_j^T \mathbf{X}}}$$

El exponente del numerador contiene la combinación lineal de las variables independientes multiplicadas por el vector de coeficientes $\mathbf{\beta}_k$, junto a su término de intercepción $\beta_{k0}$. Estos parámetros actúan como pesos analíticos que indican la influencia y dirección de cada predictor sobre la probabilidad de pertenencia a la categoría $k$.

Por su parte, el denominador representa el sumatorio de esa misma operación exponencial extendida a las $K$ categorías de la variable objetivo. Conceptualmente, el numerador se interpreta como la "puntuación cruda" de la clase $k$, mientras que el denominador actúa como un factor de normalización que escala los resultados basándose en el espacio completo de clases. Esta fracción expresa, por tanto, la probabilidad neta que el modelo asigna a que una observación específica sea clasificada dentro de la categoría $k$.


<a id="regresion_logistica_interpretabilidad"></a>
### 4.1.2 Interpretabilidad del modelo

La regresión logística multinomial se clasifica como un modelo intrínsecamente interpretable debido a que establece una relación directa y cuantificable entre los predictores y las probabilidades de clasificación a través de los coeficientes $\beta$.

La clave de su interpretabilidad radica en la transformación de sus coeficientes mediante la función exponencial ($e^{\beta_{kj}}$), lo que permite deducir las llamadas *Odds Ratios* (razones de ventajas). Esto permite realizar afirmaciones del tipo: "Por cada unidad de incremento de la variable predictora $X_j$, las probabilidades (*odds*) de que la observación sea clasificada en la clase $k$ respecto a una categoria de contraste, se multiplica por un factor $e^{\beta_{kj}}$, manteniendo constantes las demás variables del modelo"

Dado que las variables predictoras de este conjunto de datos son de naturaleza categórico binaria, la interpretación se simplifica y potencia. El incremento de una unidad representa conceptualmente la presencia (valor $1$) frente a la ausencia (valor $0$) de una característica morfológica concreta.

La magnitud y el signo del factor $e^{\beta_{kj}}$ permiten determinar si la variable $X_j$ actúa como un factor catalizador de la clase $k$ o, por el contrario, como un factor inhibidor. Esta interpretación facilita la identificación de qué variables de la morfología urbana favorecen la pertenencia de un complejo residencial a un determinado grado de cerramiento y cuáles lo evitan.

<a id="regresion_logistica_implementacion"></a>
### 4.1.3 Implementación y optimización de hiperparámetros.

La implementación práctica de la regresión logística multinomial se ha llevado a cabo mediante el estimador *LogisticRegression* de la libreria *scikit-learn*.  La estimación de los parámetros se realiza mediante el algoritmo de optimización *L-BFGS (Limited-memory Broyden–Fletcher–Goldfarb–Shanno)* puesto que se trata de un algoritmo robusto, estándar y recomendado por la librería para abordar problemas de clasificación multiclase.

Aunque el objetivo principal de este modelo es maximizar su interpretabilidad intrínseca, se ha mantenido la penalización L2 (ridge) compatible con el solver *L-BFGS*, por razones de estabilidad numérica. Esta medida es estrictamente necesaria debido a la presencia de tres variables *dummy* (DIS_1, DIS_2 y DIS_3) derivadas de la codificación *One-Hot* de la variable DIS, la cual codifica la distancia del complejo residencial al núcleo urbano principal.

La inclusión simultánea de estas tres variables, junto con el intercepto del modelo, genera de forma estructural el fenómeno de la multicolinealidad perfecta (conocido conceptualmente como la trampa de la variable ficticia). Sin una penalización, la matriz de datos se vuelve matemáticamente indeterminada, lo que provocaría que el optimizador no convergiera o, en su defecto, calculase coeficientes $\beta$ masivos o inverosímiles. La regularización L2 actúa como una medida técnica para mitigar la inestabilidad causada por este fenómeno.

Para evitar que la penalización L2 reduzca en exceso la magnitud de los coeficientes $\beta$, lo que afectaría negativamente a su interpretabilidad, se utilizan valores elevados del parámetro de regularización inversa $C$. Este parámetro controla de forma inversa la intensidad de la penalización L2, afectando al comportamiento del modelo de la siguiente manera:

* Valores bajos (ej: 0.01 o 0.1): implican una regularización fuerte, en la que el modelo prioriza la reducción de la varianza para evitar el sobreajuste, a costa de introducir un efecto de *shrinkage* que reduce la magnitud de los coeficientes $\beta$.
* Valores elevados (ej: 10 o 100): Implican una regularización reducida, permitiendo que los parámetros $\beta$ se aproximen a sus valores estadísticos insesgados.

En este sentido, el espacio de búsqueda del hiperparámetro $C$ se ha diseñado de forma deliberada para explorar valores elevados que reducen el impacto de la regularización L2 en la magnitud de los coeficientes. Esta decisión prioriza la capacidad explicativa frente a la capacidad predictiva. (citar *Shmueli, G. (2010). To explain or to predict?. Statistical Science, 25(3), 289-310*)

Esta estrategia permite que la regularización L2 opere unicamente como el soporte matemático necesario para neutralizar la multicolinealidad de las variables *dummy*, pero otorgando libertad asíntotica para que los coeficientes $\beta$ converjan lo máximo posible a sus valores estadísticos insesgados.  De este modo, la evaluación del rendimiento se realiza sobre un modelo que mantiene, en la mayor medida posible, la naturaleza explicativa que motivó su selección para este estudio.

En *scikit-learn* el valor por defecto es $C=1$ que representa una regularización moderada, por lo que se exploran valores en una escala logarítmica ascendente. La rejilla de hiperparámetros utilizada en la validación cruzada interna del esquema Nested Cross-Validation se define como:

```python
param_grid = {
    "model__C": [10, 100, 1000, 10000],
}
```

Dado que se evalúan $H=4$ combinaciones de hiperparámetros, el número total de entrenamientos realizados durante la Nested Cross-Validation se calcula como:
$$\text{Total entrenamientos} = K_{out} \times (K_{in} \times H + 1) = 5 \times (3 \times 4 +1)=65$$

Finalmente, para abordar el desbalanceo de clases presente en la variable objetivo, el estimador se configura con el parámetro interno *class_weight = "balanced"*, el cual ajusta automáticamente los pesos de cada clase de forma inversamente proporcional a su frecuencia en el conjunto de entrenamiento. De este modo, se penalizan de forma más intensa los errores en clases minoritarias, garantizando que el modelo trata de clasificar correctamente tanto las clases mayoritarias como las minoritarias.

<a id="arbol_decision"></a>
## 4.2 Árbol de decisión

<a id="arbol_decision_teoria"></a>
### 4.2.1 Fundamento téorico

Un árbol de decisión funciona mediante un proceso de partición recursiva binaria del espacio de características (variables independientes). Su objetivo es dividir dicho espacio en regiones progresivamente más homogéneas respecto a la variable objetivo. El algoritmo comienza en un nodo raiz que contiene la totalidad de las observaciones, y de forma iterativa divide los datos en grupos homogéneos llamados nodos hijos hasta llegar a una conclusión (nodo hoja) en la que no se realizan más divisiones. Cada camino desde el nodo raíz hasta un nodo hoja puede interpretarse como una regla de decisión del tipo *if-then*, formada por una conjunción de condiciones sobre las variables predictoras, que define una región del espacio de características asociada a una clase concreta.

A continuación se presenta la formulación matemática del proceso de construcción de un árbol de decisión (citar *Breiman, L., Friedman, J., Olshen, R. A., & Stone, C. J. (2017). Classification and regression trees. Chapman and Hall/CRC.*).

Sea un conjunto de datos $\mathcal{D}$ formado por $N$ muestras:
$$\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^N$$

donde:
* $\mathbf{x}_i = (x_{i1}, x_{i2}, \dots, x_{id})^T \in \mathbb{R}^d$ es el vector de características de dimensión $d$.
* $y_i \in \mathcal{Y}$ es la variable objetivo asociada a la muestra $i$.

Cada una de las $d$ características se denota como $X_j$, con $j \in \{1, 2, \dots, d\}$, representando la columna $j$ de la matriz de datos $\mathbf{x}$.

En cada nodo $m$ del árbol se selecciona una característica $X_j$ y un umbral $t$ para dividir las muestras presentes. Si denotamos como $\mathcal{D}_m \subseteq \mathcal{D}$ al subconjunto de datos que alcanza el nodo $m$, la división genera dos nuevos subconjuntos correspondientes a las ramas izquierda y derecha:

$$\mathcal{D}_{m,\text{izq}}(j, t) = \{\mathbf{x}_i \in \mathcal{D}_m : x_{ij} < t\}$$
$$\mathcal{D}_{m,\text{der}}(j, t) = \{\mathbf{x}_i \in \mathcal{D}_m : x_{ij} \ge t\}$$

La regla de decisión asociada al nodo $m$ se expresa formalmente como:
$$f_m(\mathbf{x}) = \begin{cases} m_{\text{izq}}, & \text{si } x_j < t \\ m_{\text{der}}, & \text{si } x_j \ge t \end{cases}$$

donde $x_j$ es la $j$-ésima componente del vector $\mathbf{x}$, mientras que $m_{\text{izq}}$ y $m_{\text{der}}$ representan los nodos hijos izquierdo y derecho, respectivamente, a los que se redirige la muestra.

Para decidir que característica $X_j$ y umbral $t$ produce la mejor división, el algoritmo evalúa las variables predictoras disponibles y selecciona la combinación $(X_j, t)$ que maximice la pureza de las clases en los nodos resultantes. Este proceso consiste en la minimización de una métrica de impureza. En el algoritmo CART (Classification and Regression Trees) para tareas de clasificación, la métrica estándar es el índice Gini:

$$G(m) = 1 - \sum_{k=1}^{K} p_{mk}^2$$

Donde $p_{mk}$ representa la proporción de observaciones pertenecientes a la clase $k$ dentro del nodo $m$ ($k \in \{1, \dots, K\}$).

Tras proponer una división, la impureza ponderada de los nodos hijos se calcula como:

$$G_{\text{split}}(j, t) = \frac{N_{m,\text{izq}}}{N_m} G(m_{\text{izq}}) + \frac{N_{m,\text{der}}}{N_m} G(m_{\text{der}})$$

Donde $N_m$ es el número total de muestras en el nodo $m$, mientras que $N_{m,\text{izq}}$ y $N_{m,\text{der}}$ son la cantidad de muestras asignadas a los nodos hijos izquierdo y derecho, respectivamente. La ganancia de pureza (o reducción de impureza) obtenida tras la división se define como:

$$\Delta G = G(m) - G_{\text{split}}(j, t)$$

El algoritmo selecciona el par $(X_j, t)$ que maximiza $\Delta G$, puesto que este punto de corte genera el mayor descenso neto en la impureza del sistema.

Este procedimiento se aplica de forma recursiva sobre cada nodo hijo. De este modo, el árbol construye una jerarquía donde cada nodo interno representa una condición sobre una característica y cada nodo hoja representa una predicción final (la clase mayoritaria o la distribución de probabilidad de las clases en ese subconjunto). 

El crecimiento del árbol finaliza cuando se cumple alguna condición de parada. La condición teórica o natural es que el nodo sea completamente puro (todas las observaciones pertenecen a la misma clase). Sin embargo, para evitar el sobreajuste (overfitting), es habitual controlar el crecimiento mediante hiperparámetros como la profundidad máxima del árbol, el número mínimo de muestras para dividir un nodo o un umbral mínimo para la ganancia de impureza ($\Delta G$). Una vez entrenado, el modelo clasifica una nueva observación evaluando secuencialmente las reglas de decisión desde la raíz hasta alcanzar una hoja terminal.


<a id="arbol_decision_interpretabilidad"></a>
### 4.2.2 Interpretabilidad del modelo

La interpretabilidad del árbol de decisión se fundamenta en que su estructura jerárquica puede expresarse de forma directa como un conjunto de reglas lógicas del tipo Si-Entonces (*if-then*).

Cada nodo interno del árbol representa una pregunta dicotómica sobre una variable predictora. Dado que en este conjunto de datos todas las variables son de naturaleza categórica binaria, cada división del árbol corresponde a la presencia o ausencia de una característica morfológica urbana. Esta propiedad simplifica el proceso de construcción del modelo, ya que los puntos de corte están implícitamente definidos por la propia naturaleza del dato. El algoritmo no necesita buscar heuristicamente el punto de corte óptimo.

Cada nodo hoja puede interpretarse como una regla de decisión completa, formada por la conjunción de las condiciones recorridas desde el nodo raíz, lo que permite asociar de forma explícita combinaciones de variables con una clase concreta de la variable objetivo. Por este motivo, los árboles de decisión se consideran un referente en interpretabilidad, al proporcionar una representación comprensible para expertos del dominio sin necesidad de conocimientos avanzados sobre el funcionamiento interno del modelo.


Esto permita a los expertos en el dominio disponer de multiples conjuntos de reglas, tantos como hojas terminales tenga el árbol de decisión, que le indican el conjunto de variables y las condiciones que se asocian a una determinada clase de la variable objetivo. Los árboles de decisión son la referencia en interpretabilidad puesto que este mecanismo ofrece una comprensión sencilla y directa para cualquier experto en el dominio del problema sin que necesite conocimientos avanzados sobre la algoritmia interna del modelo.

Además de esta interpretabilidad local, basada en la interpretación de las reglas que conducen a cada predicción, el árbol de decisión permite obtener una medida de interpretabilidad global mediante la cuantificación de la importancia de las variables. Esta se estima a partir de la contribución de cada variable a la reducción de la impureza a lo largo del árbol, métrica conocida como *Mean Decrease in Impurity (MDI)*.

Estas dos perspectivas de interpretabilidad  permiten, por un lado, identificar las reglas lógicas que explican la asignación de una observación a un determinado grado de cerramiento, y por otro, determinar qué variables son más relevantes en la partición del espacio de características entre las distintas clases.

<a id="arbol_decision_implementacion"></a>
### 4.2.3 Implementación y optimización de hiperparámetros

La implementación de este modelo se ha realizado mediante el estimador *DecisionTreeClassifier* de la librería *scikit-learn*, basado en el algoritmo CART (Classification and Regression Trees) propuesto por Breiman et al. (1984). Se ha seleccionado esta implementación frente a alternativas como C4.5 debido a su amplia adopción como estándar de facto en entornos de producción, así como a su integración eficiente dentro del ecosistema de *scikit-learn*, lo que facilita su combinación con el resto de componentes del framework experimental.

Al igual que en el caso de la regresión logística multinomial, la optimización del modelo no se aborda desde una perspectiva puramente predictiva. El objetivo es maximizar el rendimiento del modelo preservando su interpretabilidad intrínseca, que constituye el principal motivo de su selección en esta experimentación. En este sentido, el espacio de búsqueda de hiperparámetros se ha diseñado con el objetivo de evitar configuraciones que conduzcan a árboles excesivamente complejos, y por tanto difíciles de interpretar.

La interpretabilidad de un árbol de decisión está determinada principalmente por dos factores: la profundidad del árbol y el número de nodos hoja. La profundidad controla la longitud de las reglas de decisión, es decir, el número de condiciones encadenadas desde el nodo raíz hasta una hoja, mientras que el número de hojas determina la cantidad total de reglas del modelo. Estas dos dimensiones se controlan mediante los hiperparámetros *max_depth* y *max_leaf_nodes*, respectivamente.

La estrategia habitual consiste en limitar únicamente la profundidad máxima del árbol, manteniendo max_leaf_nodes sin restricción (None en scikit-learn). No obstante, este enfoque prioriza principalmente el rendimiento predictivo y no garantiza un control explícito sobre la complejidad global del modelo, ya que la restricción de profundidad únicamente limita la longitud de los caminos, pero no el número total de reglas. Como consecuencia, pueden generarse árboles con un elevado número de nodos hoja, lo que fragmenta el espacio de características de forma excesiva y dificulta la interpretabilidad de las reglas resultantes.

Para preservar la interpretabilidad del modelo, se propone una estrategia combinada en la que se fija la profundidad máxima del árbol para evitar reglas excesivamente largas, mientras que el espacio de búsqueda del hiperparámetro *max_leaf_nodes* se restringe. De este modo, se establece el tamaño máximo de la regla y se da libertad para que el algoritmo explore el número máximo de reglas óptimo, dentro de un espacio de búsqueda acotado y orientado a la interpretabilidad. 

La profundidad máxima del árbol se fija en max_depth = 6, mientras que el parámetro max_leaf_nodes se incorpora al proceso de búsqueda de hiperparámetros, explorando los valores $[10, 15, 20, 25, 30]$.

La representatividad de cada regla de decisión está determinada por el número de muestras asociadas al nodo hoja que representa dicha regla. El número mínimo de muestras requerido para que una hoja sea valida se controla mediante el hiperparámetro *min_samples_leaf*. Valores elevados de este parámetro contribuyen a mejorar la interpretabilidad del modelo, ya que evitan la generación de reglas excesivamente específicas basadas en subconjuntos muy pequeños de datos. El espacio de búsqueda de este hiperparámetro explora los valores $[5, 10, 15]$ fijando un mínimo de cinco muestras en cada hoja.


Por último, se considera el hiperparámetro *criterion* que controla la medida utilizada para dividir los nodos. Aunque el algoritmo CART se fundamenta en el índice de impureza de Gini, la implementación de *scikit-learn* permite utilizar la entropía como criterio de división, basada en la teoría de la información. La exploración de ambos criterios, en la optimización del modelo, permite evaluar distintas funciones de impureza y seleccionar aquellas particiones que mejor separan las clases. 

La rejilla de hiperparámetros utilizada en la validación cruzada interna del esquema Nested Cross-Validation queda definida de la siguiente forma:

```python
param_grid = {
    "model__max_leaf_nodes": [10, 15, 20, 25, 30],
    "model__min_samples_leaf": [5, 10, 15],
    "model__criterion": ["gini", "entropy"],
}
```

Dado que se evalúan $H=30$ combinaciones de hiperparámetros, el número total de entrenamientos realizados durante la Nested Cross-Validation se calcula como:
$$\text{Total entrenamientos} = K_{out} \times (K_{in} \times H + 1) = 5 \times (3 \times 30 +1)=455$$


Al igual que en el estimador *LogisticRegression*, se ha configurado el parámetro interno *class_weigh* a valor *balanced* para manejar el desbalanceo de clases. En el caso del estimador *DecisionTreeClassifier*, esta configuración ajusta internamente el cálculo de la medida de impureza, de manera que los errores en las clases menos representadas tienen un mayor impacto en la construcción del árbol.

<a id="modelos_no_interpretables"></a>
# 5. Modelos no interpretables ("caja negra")

Los modelos "caja negra" están diseñados para maximizar el rendimiento predictivo a expensas de una pérdida inherente en la interpretabilidad. Con el objetivo de explorar el límite superior de la capacidad de generalización del conjunto de datos, se han seleccionado tres algoritmos que implementan  distintas estrategias de aprendizaje no lineal.

En primer lugar, las Máquinas de Vector Soporte (SVM) con kernel de Función de Base Radial (RBF) se basan en la geometría del espacio de características, proyectando los datos a un espacio de alta dimensión. Su inclusión aporta al experimento la robustez de un clasificador basado en la maximización de márgenes, ideal para manejar fronteras de decisión complejas.

En segundo lugar, se incorporan dos métodos de *ensemble learning*: Random Forest y XGBoost. Aunque ambos algoritmos comparten el árbol de decisión como estimador base, difieren en su arquitectura. Random Forest se fundamenta en el paradigma de *bagging*, en el cual múltiples árboles se entrenan de forma independiente a partir de muestras bootstrap y sus predicciones se agregan mediante votación. Por el contrario, XGBoost se basa en *gradient boosting*, donde los árboles se construyen de manera secuencial, corrigiendo iterativamente los errores de los modelos previos mediante la optimización del gradiente. 

<a id="svm"></a>
## 5.1 Máquina de vector soporte (SVM) con kernel RBF 

<a id="svm_teoria"></a>
### 5.1.1 Fundamento teórico

El objetivo fundamental de una Support Vector Machine (SVM) (presentada originalmente por *Vapnik, V. (1995). Support-vector networks. Machine learning, 20, 273-297*) es encontrar un hiperplano de separación que maximice la distancia (margen) entre las muestras de distintas clases. Para un conjunto de datos $(x_i,y_i)$, donde $x_i$ representa las características de entrada e $y_i \in [-1,1]$ es la etiqueta de clase, el hiperplano se define como:
​
$$w^Tx+b=0$$

donde $w$ es el vector normal al hiperplano y $b$ es el término de sesgo. En el caso ideal de datos linealmente separables, SVM busca el hiperplano que maximiza el margen entre las clases, formulando el problema como una tarea de optimización. Sin embargo, en muchos problemas reales las clases no pueden separarse mediante una frontera lineal. Para superar esta limitación, SVM emplea el denominado *kernel trick*, que permite transformar implícitamente los datos a un espacio de características de mayor dimensión en el que la separación entre clases puede resultar más sencilla.

Uno de los kernels más utilizados, y el empleado en este experimento, es el kernel de función base radial (*Radial Basis Function, RBF*), definido como:

$$K(\mathbf{x}_i, \mathbf{x}_j) = \exp\left(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2\right)$$

donde $\mathbf{x}_i$ y $\mathbf{x}_j$ son los vectores de características de dos muestras. Esta función mide el grado de similitud entre dos vectores de características al calcular la norma euclídea al cuadrado. En este conjunto de datos todas las variables predictoras son categorico binarias, por lo que el calculo de $\|\mathbf{x}_i - \mathbf{x}_j\|^2$ sobre vectores binarios es el equivalente a la distancia de Hamming. Esta distancia puede interpretarse en este problema, como el número de diferencias morfológicas que tienen dos complejos residenciales. 

La exponencial negativa transforma la distancia entre dos muestras en una medida de similitud acotada en el intervalo $[0,1]$, donde valores próximos a $1$ indican una alta similitud y valores cercanos a $0$ indican una baja similitud. El hiperparámetro $\gamma$ controla la rapidez con la que esta similitud disminuye a medida que aumenta la distancia entre las muestras.

Valores bajos de $\gamma$ hacen que la similitud decrezca lentamente, de modo que muestras relativamente alejadas siguen ejerciendo influencia sobre la frontera de decisión. Como consecuencia, el modelo genera fronteras más suaves y globales, lo que puede provocar *underfitting* si no es capaz de capturar adecuadamente la estructura de los datos. Por el contrario, valores elevados de $\gamma$ hacen que la similitud disminuya rápidamente incluso ante pequeñas diferencias entre muestras. En este caso, la influencia de cada vector de soporte queda restringida a una región muy local del espacio de características, permitiendo construir fronteras de decisión más complejas. Aunque esto aumenta la capacidad de adaptación del modelo a los datos de entrenamiento (*overfitting*).

Mediante este kernel, el algoritmo opera implícitamente en un espacio de características de dimensión muy elevada, donde busca el hiperplano que maximiza el margen de separación entre las clases. Durante el proceso de entrenamiento, el algoritmo identifica las muestras más relevantes para definir la frontera de decisión, conocidas como vectores soporte. Estas muestras se determinan a través de los multiplicadores de Lagrange $\alpha_i$: aquellas para las que $\alpha_i > 0$ se convierten en vectores soporte, mientras que las muestras con $\alpha_i = 0$ quedan fuera de la función de decisión al encontrarse suficientemente alejadas de la frontera.

Una vez entrenado el modelo, la predicción de una nueva muestra $\mathbf{x}$ se realiza mediante la siguiente función de decisión, que únicamente involucra a los vectores soporte:

$$f(\mathbf{x}) = \sum_{i \in \text{Vectores de Soporte}} \alpha_i y_i K(\mathbf{x}_i, \mathbf{x}) + b$$

Conceptualmente, esta expresión calcula la similitud entre la nueva muestra y cada uno de los vectores soporte mediante el kernel RBF. Cada vector soporte contribuye a la decisión con una intensidad determinada por $\alpha_i$ y el signo asociado a su clase. Finalmente, la combinación de todas estas contribuciones determina la clase asignada a la nueva muestra.

Aunque SVM fue concebido originalmente para problemas de clasificación binaria, puede extenderse a escenarios multiclase mediante estrategias de descomposición. Las más habituales son:
* One-vs-One (OvO): se entrena un clasificador para cada pareja de clases y la predicción final se obtiene mediante un sistema de votación.
* One-vs-Rest (OvR): se entrena un clasificador binario para cada clase, considerando dicha clase como positiva y agrupando el resto de clases en una única categoría negativa. Durante la inferencia, cada clasificador genera una puntuación que refleja el grado de pertenencia de la observación a la clase correspondiente. La predicción final se obtiene asignando la muestra a la clase asociada a la mayor puntuación.

<a id="svm_interpretabilidad"></a>
### 5.1.2 Limitaciones de interpretabilidad

el clasificador SVM con kernel RBF carece de interpretabilidad intrínseca debido a la transformación matemática que induce el *kernel trick*. En una SVM lineal, el modelo conserva cierto grado de interpretabilidad, puesto que las decisiones se realizan mediante una función de la forma:
$$f(x)=w^Tx+b$$

donde cada componente $w_j$ del vector de pesos está asociado a una variable $x_j$. Esto permite realizar un análisis de importancia de características, donde la magnitud del peso representa la relevancia de la variable y su signo indica la dirección del impacto hacia una de las clases.

Sin embargo, al introducir el kernel RBF, el algoritmo abandona el espacio original de los datos para operar de forma implícita en un espacio latente de alta dimensión. Como consecuencia, desaparece la relación directa entre las variables predictoras y la función de decisión. En lugar de depender de coeficientes asociados a cada variable, la clasificación se determina a partir de la similitud geométrica de la observación con los vectores de soporte. Esto dificulta la interpretación del modelo, ya que no es posible identificar de forma directa cómo contribuye cada variable individual a la predicción.

Por este motivo, un modelo SVM con kernel RBF es clasificado como "caja negra" y es necesario usar herramientas *post-hoc* para interpretar su comportamiento.

<a id="svm_implementacion"></a>
### 5.1.3 Implementación y optimización de hiperparámetros

La implementación práctica del modelo SVM con kernel RBF se ha llevado a cabo mediante el estimador *SVC* de la libreria *Scikit-learn*. El parámetro *kernel* queda configurado a valor 'rbf' para habilitar el uso de la función base radial como proyeción a un espacio de alta dimensión.

Al tratarse de un problema de clasificación multiclase, se configura el parámetro *decision_function_shape* a valor 'ovr' que especifica que se implemente la estrategia de descomposición *One-vs-Rest* en la que se entrena un clasificador para cada clase produciendo una función de decisión propia. Para una nueva muestra $x$ se evaluan todos los clasificadores y se asigna la clase que haya obtenido un mayor *score*.

La optimización del modelo mediante la búsqueda en rejilla se centra en los dos hiperparámetros que gobiernan el funcionamiento del kernel RBF.
* Parámetro de regularización C: Controla el equilibrio entre la maximización del margen de separación y la tolerancia a los errores de clasificación en las muestras de entrenamiento. Valores bajos priorizan un margen más amplio a costa de permitir más errores (modelo más regularizado) mientras que valores altos imponen una penalización más severa al error. En el espacio de búsqueda se considera una escala logaritmica desde valores bajos hasta valores altos: $[0.1, 1, 10, 100]$.
* Parámetro del radio del kernel ($\gamma$). Regula la curvatura de la frontera de decisión al controlar el radio de influencia de los vectores de soporte como se ha comentado en la sección anterior. Ademas de valores numéricos especificos ($0.01$ y $0.1$) se incluyen las heurísticas estándar de *scikit-learn*: 'scale' y 'auto'. La primera adapta el radio en función de la varianza de las características mientras que 'auto' escala unicamente utilizando el número de características. 

La rejilla de hiperparámetros utilizada en la validación cruzada interna del esquema Nested Cross-Validation queda definida de la siguiente forma:

```python
param_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__gamma": ["scale", "auto", 0.01, 0.1]
}
```

Dado que se evalúan $H=16$ combinaciones de hiperparámetros, el número total de entrenamientos realizados durante la Nested Cross-Validation se calcula como:
$$\text{Total entrenamientos} = K_{out} \times (K_{in} \times H + 1) = 5 \times (3 \times 16 +1)=245$$

Finalmente, para manejar el desbalanceo de clases se ha configurado el parámetro *class_weight* del estimador a valor *balanced*.

<a id="random_forest"></a>
## 5.2 Random Forest

<a id="random_forest_teoria"></a>
### 5.2.1 Fundamento teórico

Random Forest es un método de ensemble learning que combina múltiples árboles de decisión con el objetivo de reducir la varianza, mejorar la capacidad de generalización y aumentar el rendimiento predictivo. El algoritmo fue propuesto por Breiman (2001) y se fundamenta en dos mecanismos complementarios para introducir diversidad entre los árboles que componen el conjunto: el muestreo bootstrap (bagging) y la selección aleatoria de características.

Sea $D$ un conjunto de datos compuesto por $N$ observaciones. Para cada árbol $b$, donde $(b = 1,\ldots,B)$, se genera un conjunto de entrenamiento $D_b$ mediante muestreo aleatorio con reemplazo a partir de $D$. Todos los subconjuntos de entrenamiento $D_b$ contienen $N$ observaciones, sin embargo, algunas pueden aparecer varias veces, mientras que otras pueden no ser seleccionadas. Esto es lo que se denomina *Bootstrap sampling* y provoca que cada árbol "vea" una composición ligeramente distinta del conjunto de datos original.

La probabilidad de que una observación concreta no sea incluida en una muestra bootstrap viene dada por:

$$\left(1 - \frac{1}{N}\right)^N \approx e^{-1} \approx 0.368$$

Por tanto, cada árbol se entrena utilizando aproximadamente un $63.2\%$ de observaciones únicas del conjunto original, mientras que el $36.8\%$ restante queda fuera de la muestra de entrenamiento. Estas observaciones reciben el nombre de datos *Out-of-Bag (OOB)* y pueden utilizarse para estimar internamente el error de generalización del modelo.

Aunque el muestreo bootstrap introduce diversidad entre los árboles, esta suele ser insuficiente cuando existen variables predictoras especialmente dominantes. Para incrementar la decorrelación entre árboles, Random Forest incorpora un segundo mecanismo basado en la selección aleatoria de características. 

Mientras que un árbol de decisión convencional evalúa las $M$ variables predictoras disponibles para determinar la partición óptima de cada nodo, Random Forest restringe esta búsqueda a un subconjunto aleatorio de tamaño $m$, donde habitualmente se emplea la heurística $m=\sqrt{M}$. En cada nodo únicamente se consideran las variables incluidas en dicho subconjunto aleatorio para calcular el criterio de impureza y seleccionar la mejor división. Este procedimiento evita que las variables más influyentes aparezcan sistemáticamente en los primeros niveles de todos los árboles, favoreciendo la generación de modelos menos correlacionados y capaces de capturar patrones más complejos.

Una vez entrenados los $B$ árboles que componen el bosque, la clasificación de una nueva observación $\mathbf{x}$ se obtiene mediante un esquema de votación por mayoría. Cada árbol emite una predicción independiente y la clase final $\hat{y}$ corresponde a aquella que recibe el mayor número de votos:
$$
\hat{y}=\arg\max_{k \in {1,\ldots,K}}
\sum_{b=1}^{B}
\mathbb{I}\left(h_b(\mathbf{x})=k\right)
$$
donde $h_b(\mathbf{x})$ representa la predicción del árbol $b$ y $\mathbb{I}(\cdot)$ es la función indicadora.

<a id="random_forest_interpretabilidad"></a>
### 5.2.2 Limitaciones de interpretabilidad

Aunque Random Forest está compuesto por árboles de decisión, que de forma individual son modelos altamente interpretables, el conjunto resultante se considera un modelo de "caja negra" debido a la forma en equ obtiene sus predicciones. 

Como se ha descrito en la sección anterior, cada árbol se construye a partir de una muestra bootstrap diferente y emplea subconjuntos aleatorios de variables durante el proceso de partición. Como consecuencia, cada árbol aprende reglas de decisión distintas que pueden solaparse o incluso ser parcialmente inconsistentes entre sí.

La clasificación final se obtiene mediante la combinación de las predicciones de todos los árboles mediante un mecanismo de votación por mayoría. Por tanto, la decisión del modelo no puede explicarse a través de una única secuencia de reglas lógicas, sino que emerge de la interacción de cientos o incluso miles de reglas distribuidas entre los distintos árboles del bosque que pueden solaparse o incluso ser parcialmente inconsistentes entre sí.

Esto provoca que la interpretación del modelo a través de las reglas generadas sea iniviable en la práctica y se necesitre recurrir a técnicas de interpretación *post-hoc*.

<a id="random_forest_implementacion"></a>
### 5.2.3 Implementación y optimización de hiperparámetros

La implementación práctica del algoritmo *Random Forest* se ha llevado a cabo mediante el estimador *RandomForestClassifier* de la libreria *scikit-learn*. A diferencia del árbol de decisión individual, donde el espacio de búsqueda de hiperparámetros se diseñó con el objetivo de preservar la interpretabilidad del modelo, en este caso la optimización se orienta exclusivamente a maximizar el rendimiento predictivo. Para ello, se define una rejilla de hiperparámetros suficientemente amplia que permite explorar diferentes configuraciones.

Los hiperparámetros considerados son los siguientes:

* **Número de árboles (n_estimators)**. Este parámetro determina el número de árboles que componen el ensemble. En general, un mayor número de árboles reduce la varianza del modelo y aumenta la estabilidad de las predicciones, aunque a partir de cierto punto las mejoras tienden a estabilizarse. Se exploran los valores $[100, 200, 300]$, considerados suficientes para aproximarse al régimen de convergencia estadística del algoritmo.
* **Subespacio de variables (max_features)**. Controla el tamaño del subconjunto aleatorio de variables que puede utilizarse para determinar la mejor partición en cada nodo. Dado que el conjunto de datos dispone de 25 variables predictoras, se exploran los valores $['sqrt', 0.3, 0.4]$, que permiten evaluar $5$, $7$ y $10$ variables por nodo, respectivamente.
* **Profundidad máxima (max_depth)**. A diferencia de la configuración realizada en el  árbol de decisión, se incluye el valor "None" que permite que los arboles crezcan hasta la maxima pureza posible. Se complementa la búsqueda de este hiperparámetro con valores de profundidad altos $5$ y $10$ para evaluar si una ligera poda mejora el rendimiento.
* **Número mínimo de muestras para dividir un nodo (min_samples_split)**.Este hiperparametro define el número mínimo de observaciones requerido para que un nodo pueda subdividirse. Se consideran los valores $[2, 5]$, permitiendo comparar una estrategia de crecimiento prácticamente sin restricciones con otra ligeramente más conservadora.
* **criterio de partición (criterion)**. Al igual que en el árbol de decisión se exploran tanto 'gini' como 'entropy'.

La rejilla de hiperparámetros utilizada en la validación cruzada interna del esquema Nested Cross-Validation queda definida de la siguiente forma:
```python
param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [5, 10, None],
    "model__max_features": ["sqrt", 0.3, 0.4],
    "model__criterion": ['gini', 'entropy'],
    "model__min_samples_split": [2, 5],
}
```
Dado que se evalúan $H=108$ combinaciones de hiperparámetros, el número total de entrenamientos realizados durante la Nested Cross-Validation se calcula como:
$$\text{Total entrenamientos} = K_{out} \times (K_{in} \times H + 1) = 5 \times (3 \times 108 +1)=1625$$

Para manejar el desbalanceo de clases se ha configurado el parámetro interno *class_weight* a valor *"balanced"*. Al igual que en el árbol de decisión, esta configuración ajusta automáticamente los pesos de las clases de forma inversamente proporcional a su frecuencia en los datos de entrenamiento. Como consecuencia, el cálculo de las medidas de impureza empleadas durante la construcción de los árboles tiene en cuenta dichos pesos, incrementando la penalización asociada a los errores cometidos sobre las clases minoritarias y favoreciendo un tratamiento más equilibrado de todas las categorías durante el proceso de aprendizaje.

<a id="xgboost"></a>
## 5.3 XGBoost (Extreme Gradient Boosting)

<a id="xgboost_teoria"></a>
### 5.3.1 Fundamento teórico

A diferencia de Random Forest, donde los árboles se entrenan de manera independiente y en paralelo, el algoritmo XGBoost (Extreme Gradient Boosting) se basa en la construcción secuencial de un conjunto de árboles de decisión. La idea fundamental de este algoritmo es que cada nuevo árbol se entrena para corregir los errores cometidos por los árboles previamente generados, mejorando progresivamente la capacidad predictiva del modelo (citar *Chen, T., & Guestrin, C. (2016, August). Xgboost: A scalable tree boosting system.*).

Para ello, en cada iteración $t$, el algoritmo incorpora un nuevo árbol de decisión $f_t$ y trata de minimizar la siguiente función objetivo:
$$\mathcal{L}^{(t)} = \underbrace{\sum_{i=1}^N l\left(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)\right)}_{\text{Pérdida de entrenamiento}} + \underbrace{\Omega(f_t)}_{\text{Regularización}}$$

Esta función objetivo consta de dos componentes: un término de pérdida y un término de regularización. El objetivo es que el nuevo árbol mejore el rendimiento predictivo del conjunto de árboles existente, evitando que su estructura sea excesivamente complejos y propensa al sobreajuste.
 
La función de pérdida de entrenamiento cuantifica el error entre la etiqueta real $y_i$ y la nueva predicción del modelo, obtenida al sumar la contribución del árbol $f_t(x_i)$ a la predicción acumulada de las iteraciones anteriores $\hat{y_i}^{(t-1)}$.

La función de regularización $\Omega(f_t)$ es la penalización aplicada al nuevo árbol $f_t$, cuyo objetivo es controlar la complejidad del modelo y reducir el riesgo de sobreajuste. Este término se define como:
$$\Omega(f_t) = \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2$$
donde:
* $T$: Es el número total de hojas que tiene el nuevo árbol.
* $\gamma$ (Gamma): Es un hiperparámetro que penaliza la creación de nuevas hojas. Cuanto mayor sea su valor, más conservador será el crecimiento del árbol.
* $w_j$: es el peso o score asociado a la hoja $j$, que determina la contribución de dicha hoja a la predicción final.
* $\lambda$ (Lambda): Es el coeficiente de regularización L2 aplicado a los pesos de las hojas. Este hiperparámetro favorece valores de peso más pequeños, reduciendo la sensibilidad del modelo a variaciones en los datos y mejorando su capacidad de generalización.

La optimización directa de esta función objetivo resulta computacionalmente costosa. Para abordar este problema, los autores de XGBoost emplean una aproximación de Taylor de segundo orden que permite reformular está función en una expresión más manejable que depende únicamente de las derivadas de primer y segundo orden de la función de pérdida:
$$\tilde{\mathcal{L}}^{(t)} = \sum_{i=1}^N \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2$$

En esta expresión aparecen dos términos fundamentales en XGBoost que permiten que cada nuevo árbol corrija de forma iterativa los errores acumulados por los árboles construidos en iteraciones anteriores:
* **gradiente ($g_i$)** El gradiente es la primera derivada de la función de pérdida. Este término indica la dirección y la magnitud del error cometido en cada observación, proporcionando información sobre cómo debe modificarse la predicción para reducir dicho error.
* **hessiano ($h_i$)**. El hessiano es la segunda derivada de la función de pérdida y representa la curvatura de esta función. Conceptualmente actúa como un peso que indica con que velocidad hay un cambio en el error conforme se ajustan las predicciones.

Los árboles de decisión de XGBoost no realizan las particiones en los nodos con el objetivo de dividir los datos respecto a la clase, su objetivo es encontrar el punto de corte que maximice la ganancia al agrupar muestras con gradientes y hessianos similares. Por esta razón, XGBoost no emplea métricas como la impureza de Gini o la entropía. En su lugar, define una medida propia de ganancia basada en la reducción de la función objetivo. Esta métrica cuantifica cuánto mejora el modelo al realizar una determinada partición y se expresa de la siguiente forma:

$$\text{Gain} = \frac{1}{2} \left[ \underbrace{\frac{\left(\sum_{i \in I_L} g_i\right)^2}{\sum_{i \in I_L} h_i + \lambda}}_{\text{Calidad del nodo hijo izquierdo}} + \underbrace{\frac{\left(\sum_{i \in I_R} g_i\right)^2}{\sum_{i \in I_R} h_i + \lambda}}_{\text{Calidad del nodo hijo derecho}} - \underbrace{\frac{\left(\sum_{i \in I} g_i\right)^2}{\sum_{i \in I} h_i + \lambda}}_{\text{Calidad del nodo original}} \right] - \gamma$$

donde:
* $I$: conjunto de observaciones que pertenecen al nodo antes de realizar la partición.
* $I_L$: conjunto de observaciones que quedan en el nodo hijo izquierdo tras la división.
* $I_R$: conjunto de observaciones que quedan en el nodo hijo derecho.
* $λ$: parámetro de regularización L2 aplicado a los pesos de las hojas.
* $γ$ : penalización por crear una nueva división en el árbol. Este hiperparámetro evita que el algoritmo genere divisiones que aporten una mejora insignificante. Si la ganancia obtenida no supera γ, la partición se descarta.


Esta función de ganancia compara la calidad de los dos nodos hijos con la calidad del nodo original. Si la suma de las contribuciones de los nodos izquierdo y derecho supera suficientemente a la del nodo padre, la ganancia será positiva y la división será aceptada. En caso contrario, el nodo permanecerá sin dividir. Este mecanismo permite que XGBoost solo cree una nueva rama si la reducción esperada del error compensa la complejidad adicional introducida por la nueva partición.

Una vez introducidos los conceptos fundamentales en los que se basa XGBoost, puede describirse el funcionamiento iterativo del algoritmo considerando inicialmente un problema de clasificación binaria. En problemas de clasificación multiclase, como el abordado en este trabajo, XGBoost extiende este planteamiento mediante una estrategia One-vs-Rest (OvR), mediante la optimización de una función de pérdida multinomial basada en softmax. No obstante, el mecanismo de construcción y mejora iterativa de los árboles sigue siendo esencialmente el mismo.

Sea un conjunto de datos $D = \{(x_i, y_i)\}_{i=1}^N$ (donde $y_i \in \{0, 1\}$), XGBoost no resuelve el problema de clasificación binaria prediciendo directamente los valores de la clase, sino que predice un valor numérico denominado log-odd ($\hat{y}$) o predicción cruda. Por tanto, aunque se trate de un problema de clasificación, los árboles que componen el modelo se comportan como árboles de regresión, ya que generan valores continuos que posteriormente son transformados en probabilidades. 

En la iteración $t$, el nuevo árbol $f_t$ se construye a partir del estado actual del modelo, determinado por la suma de las predicciones generadas (*log-odss*) por todos los árboles construidos hasta la iteración anterior, denotada por $\hat{y}^{(t-1)}$. A partir de estas predicciones acumuladas, se calculan para cada muestra $x_i$, el gradiente $g_i$ y el hessiano $h_i$ de la función de pérdida. Con estos valores, el algoritmo construye el nuevo árbol de decisión evaluando, en cada nodo, las posibles particiones del espacio de características y seleccionando aquella que maximiza la ganancia definida anteriormente.

Tras completar la estructura del árbol $f_t$, cada observación $x_i$ queda asignada a una hoja terminal. En cada hoja se calcula el score o peso óptimo ($w_j^*$), obtenido a partir de la minimización de la función objetivo aproximada mediante la expansión de Taylor de segundo orden:
$$w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}$$

donde $I_j$ es el conjunto de datos que pertenecen a la hoja $j$. Esta expresión muestra que el peso asignado a cada hoja depende directamente de los gradientes y hessianos acumulados de las muestras que pertenecen a esa hoja. Intuitivamente, el numerador determina la dirección y magnitud de la corrección necesaria para reducir el error, mientras que el denominador modera dicha corrección teniendo en cuenta la curvatura de la función de pérdida y el efecto de la regularización L2 controlada por el parámetro $\lambda$.

Finalmente, el nuevo árbol de decisión genera una corrección sobre la predicción actual del modelo. Para cada observación $x_i$, dicha corrección viene determinada por el peso óptimo de la hoja en la que ha quedado asignada, es decir, $f_t(x_i)=w_j^*$.

La predicción cruda (*log-odd*) en la iteración $t$ se obtiene sumando la contribución del nuevo árbol a la predicción acumulada hasta la iteración anterior:
$$\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + \eta f_t(x_i)$$

donde $\eta$ es la tasa de aprendizaje (learning rate). Este hiperparámetro controla la magnitud de la contribución de cada nuevo árbol al modelo final. Valores pequeños de $\eta$ hacen que las actualizaciones sean más graduales, reduciendo el riesgo de sobreajuste y favoreciendo una mejor capacidad de generalización. Pero, suelen requerir un mayor número de árboles para alcanzar un alto nivel de rendimiento.

De esta forma, cada árbol no intenta resolver el problema de clasificación por sí mismo, sino que aporta una pequeña corrección a las predicciones acumuladas del conjunto. Mediante este proceso iterativo, el modelo va reduciendo progresivamente los errores residuales hasta converger hacia una solución que minimiza la función objetivo.


Una vez entrenados los $M$ árboles que componen el ensemble, la predicción final se obtiene a partir de la suma acumulada de las contribuciones de todos los árboles. Esta predicción cruda se transforma en una probabilidad mediante la función sigmoide, en el caso de la clasificación binaria:
$$P(y = 1 | x) = \frac{1}{1 + e^{-\sum_{t=1}^M \eta f_t(x)}}$$

La probabilidad obtenida se compara con un umbral de decisión, que por defecto suele fijarse en $0.5$. Si $P(y=1 \mid x) > 0.5$, la observación se clasifica como perteneciente a la clase positiva ($y=1$); en caso contrario, se asigna a la clase negativa $(y=0)$.

En problemas de clasificación multiclase, el procedimiento es análogo, aunque la función sigmoide se sustituye por la función softmax, que permite obtener una distribución de probabilidad sobre todas las clases posibles y seleccionar aquella con la probabilidad más elevada.

<a id="xgboost_interpretabilidad"></a>
### 5.3.2 Limitaciones de interpretabilidad

El modelo XGBoost se considera un modelo de tipo caja negra debido a la complejidad del ensemble aditivo que lo compone. La predicción final se obtiene como la suma de las contribuciones de cientos de árboles de decisión construidos de forma secuencial, donde cada nuevo árbol se entrena para corregir los errores residuales del modelo en la iteración anterior.

Además, en este algoritmo los árboles de decisión no son interpretables de forma independiente. La estructura de cada árbol en XGBoost se construye mediante particiones que maximizan la reducción de la función objetivo, por lo que las reglas generadas no pueden interpretarse ya que no tienen relación directa con la variable objetivo.

Como consecuencia, XGBoost ofrece un alto rendimiento en problemas con datos tabulares, pero presenta una interpretabilidad intrínseca prácticamente inexistente. Por este motivo, su análisis requiere el uso de técnicas de interpretación post-hoc que permitan aproximar o explicar su comportamiento.

<a id="xgboost_interpretabilidad"></a>
### 5.3.3 Implementación y optimización de hiperparámetros

La implementación práctica del algoritmo XGBoost se ha llevado a cabo mediante el estimador XGBClassifier, integrado en la librería de código abierto xgboost. Aunque es una libreria independiente de *scikit-learn*, cuenta con una interfaz adaptada que permite integrarla de forma nativa con el *framework* del experimento.

Dado que se trata de un problema de clasificación multiclase, se han configurado los siguientes parámetros estructurales del estimador:
* *objective = 'multi:softmax'*. Define el uso de una función de pérdida multinomial basada en softmax para problemas de clasificación multiclase. Bajo esta configuración, el modelo devuelve directamente la clase predicha.
* *eval_metric = 'mlogloss'*. Utiliza la log-loss multiclase (entropía cruzada) como métrica de evaluación del modelo durante el entrenamiento.

La librería XGBoost incorpora mecanismos de submuestreo inspirados en técnicas como Bagging con el objetivo de reducir el riesgo de sobreajuste y mejorar la generalización del modelo. Este submuestreo puede aplicarse tanto a las observaciones como a las variables.

El submuestreo de observaciones se controla mediante el hiperparámetro subsample, que especifica la fracción de muestras utilizada en cada iteración para el entrenamiento del árbol. Este proceso introduce aleatoriedad en la construcción del modelo y contribuye a reducir la correlación entre árboles.  El tipo de submuestreo se configura con el parámetro *sampling_method* en el que se ha establecido el modo *uniform* para garantizar que todas las muestras tienen las mismas probabilidades de ser seleccionadas.


Por otro lado, el submuestreo de las variables inspirado en *Random Forest*, puede llevarse a cabo desde distintos enfoques a través de los siguientes parámetros:
* *colsample_bytree*. selecciona aleatoriamente un subconjunto de variables antes de construir cada árbol, manteniéndolo fijo durante todo su crecimiento.
* *colsample_bylebel*. aplica el submuestreo de variables en cada nivel de profundidad del árbol, de forma que el conjunto de variables disponibles puede variar entre niveles.
* *colsample_bynode*. realiza un submuestreo independiente de variables en cada nodo antes de evaluar las particiones.

Se ha optado por colsample_bytree, ya que proporciona un nivel adecuado de aleatoriedad a nivel de árbol completo sin introducir una variabilidad excesiva en la estructura interna del mismo. Dado que el número de variables del conjunto de datos no es elevado, no se considera necesario aplicar submuestreos más agresivos por nivel o por nodo, como en colsample_bylevel o colsample_bynode, que podrían incrementar la inestabilidad del modelo sin aportar mejoras significativas en este contexto.

Al igual que en Random Forest, el espacio de búsqueda de hiperparámetros se ha diseñado con el objetivo de maximizar el rendimiento predictivo. Para ello, se define una rejilla de hiperparámetros suficientemente amplia que permite explorar diferentes configuraciones.

Los hiperparámetros considerados son los siguientes:

* **Número de árboles (n_estimators)**. Este parámetro determina el número de iteraciones del proceso de boosting, es decir, el número de árboles que componen el ensemble. Se exploran los valores $[100,300]$, que permiten comparar un escenario con un número elevado de iteraciones frente a una configuración más limitada.
* **Tasa de aprendizaje (leraning_rate)**. Este hiperparámetro controla la contribución de cada árbol al modelo final, regulando la velocidad de aprendizaje del algoritmo. Existe una relación inversa entre este parámetro y el número de árboles, de forma que valores más pequeños requieren un mayor número de árboles para alcanzar un nivel de convergencia equivalente. Por ello, se exploran los valores $0.01$ y $0.1$ que se complementan con los valores seleccionados en *n_estimators*.
* **Profundidad máxima (max_depth)**. Determina la profundidad máxima de cada árbol. Se consideran los valores $5$ y $10$, que representan niveles bajo y alto de complejidad estructural. En este algoritmo la propia complejidad esta controlada en la función objetivo mediante el término de regularización.
* **Submuestreo de las observaciones (subsample)**. Controla la fracción de muestras utilizadas en la construcción de cada árbol. Se evalúan los valores $0.8$ y $1$, comparando un escenario con introducción de aleatoriedad frente a otro determinista equivalente al boosting clásico.
* **Submuestreo de las variables (colsample_bytree)**. Define la proporción de variables seleccionadas aleatoriamente en la construcción de cada árbol. Se exploran los valores 0.8 y 1, con el mismo proposito que en el submuestreo de observaciones.
* **Regularización L2 (lambda)**. Controla la penalización sobre los pesos de las hojas, reduciendo la magnitud de los coeficientes y limitando la complejidad del modelo. Se emplea el valor por defecto $\lambda=1$ junto con un valor más elevado $\lambda=5$, que incrementa la restricción sobre los pesos y favorece soluciones más conservadoras.
* **Umbral de penalización por partición (gamma)**. Este hiperparámetro controla la ganancia mínima requerida para realizar una nueva partición en un nodo. Se exploran los valores $0$ y $0.3$, comparando un escenario sin restricción con otro regularizado.

La rejilla de hiperparámetros utilizada en la validación cruzada interna del esquema Nested Cross-Validation queda definida de la siguiente forma:
```python
param_grid = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [5, 10],
    "model__learning_rate": [0.01, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
    "model__gamma": [0, 0.3],
    "model__reg_lambda": [1, 5]
}
```
Dado que se evalúan $H=128$ combinaciones de hiperparámetros, el número total de entrenamientos realizados durante la Nested Cross-Validation se calcula como:
$$\text{Total entrenamientos} = K_{out} \times (K_{in} \times H + 1) = 5 \times (3 \times 128 +1)=1925$$

A diferencia de otros modelos empleados en el experimento, XGBoost no incorpora un mecanismo equivalente a class_weight="balanced" para el tratamiento directo del desbalanceo de clases en problemas multiclase. Por este motivo, se ha optado por calcular de forma externa la ponderación de las clases.

Para ello se utiliza la función *compute_sample_weights()* del módulo *class_weights()* de la libreria *scikit-learn*, aplicada sobre el conjunto de entrenamiento de cada partición (*Outer Train Set*) dentro del esquema Nested Cross Validation. Este procedimiento garantiza que los pesos se estiman exclusivamente a partir de los datos de entrenamiento, evitando cualquier tipo de data leakage. Posteriormente, dichos pesos se incorporan como parámetro en el proceso de entrenamiento de XGBoost, de forma que el algoritmo penaliza de manera diferenciada los errores en función de la clase asociada a cada observación.

La implementación de este procedimiento se realiza en la función *performance_experiment()*, que se describe en la siguiente sección.

<a id="implementacion_experimental"></a>
# 6. Implementación experimental

La implementación experimental de la evaluación de rendimiento se ha llevado a cabo mediante el desarrollo de la función *performance_experiment()* cuyo código se encuentra en *src/performance_utils.py*.

Esta función recibe los siguientes parámetros de entrada:
* **config**. Objeto de configuración que contiene las rutas para leer el conjunto de datos y los ficheros que contienen los *folds* para la validación cruzada externa (*Outer Loop*) e interna (*Inner Loop*).
* **pipeline**. Objeto pipeline de la clase Pipeline de *scikit-learn* que contiene el preprocesamiento y el estimador (modelo) que se desea evaluar.
* **param_grid**. Diccionario que define el espacio de búsqueda de hiperparámetros para el estimador contenido en el *Pipeline*.
* **experiment_name**. Nombre que se asigna al experimento y que se utilizará para almacenar los ficheros de resultados.

Además se definen dos parámetros opcionales:
* **gridsearch_metric**. Métrica objetivo para la optimización del modelo durante la búsqueda de hiperparámetros. El valor por defecto es "f1_macro" puesto que es la métrica objetivo definida para esta experimentación.
* **use_balanced_weights**. Valor booleano que indica si se calculan los pesos de cada clase con la función *compute_sample_weights()* y se aplican en el entrenamiento. Por defecto está a valor "False", unicamente se utiliza con el algoritmo XGBoost ya que la implementación en *scikit-learn* del resto de algoritmos evaluados incorpora un mecanismo interno para manejar el desbalanceo de clases mediante el parámetro *class_weight*.

Aunque la función se ha diseñado para aceptar un objeto pipeline, en este conjunto de datos todas las variables predictoras son de tipo categórico binario, codificadas con valores 0 y 1. Por este motivo, no es necesario aplicar ningún tipo de escalado, y el pipeline se compone únicamente del estimador que define el modelo correspondiente.

La función realiza el siguiente procedimiento:
1. Carga el conjunto de datos.
2. Carga las particiones predefinidas de la validación cruzada externa (*Outer CV*) almacenadas en el fichero CSV correspondiente.
3. Para cada *Outer fold*:
    * Construye los conjuntos de entrenamiento (*Outer Train Set*) y test (*Outer Test Set*).
    * Calcula los pesos de cada clase si *use_balanced_weights=True*.
    * Carga las particiones predefinidas de la validación cruzada interna (*Inner CV*), almacenadas en el fichero CSV correspondiente a la partición externa (*Outer fold*) actual.
    * Construye *Inner Train Set* e *Inner Validation Set*.
    * Validación cruzada interna (*INNER CV*): Ejecuta el proceso de búsqueda de hiperparámetros mediante GridSearchCV.
    * Selecciona la configuración óptima de hiperparámetros
    * Entrena el modelo óptimo sobre el *Outer Train Set*.
    * Evalua el modelo entrenado sobre *Outer Test Set*.
    * Cálcula las métricas globales (F1-score Macro, MCC y *accuracy*) del *Outer fold* actual.
    * Almacena las predicciones *Out-of-fold* y las etiquetas de clase de las muestras de *Outer Test Set*.
4. Almacena en un único *dataframe* los resultados de las métricas globales en cada fold.
5. Genera la matriz de confusión a partir de todas las predicciones *Out-of-fold*.
6. Calcula las métricas específicas por clase (precision, recall y F1-score).
7. Devuelve los resultados en *dataframes* y los almacena en ficheros CSV en la carpeta *output/5x3_NCV/experiment_name*.

En el siguiente diagrama de bloque se ilustra el procedimiento descrito:
<img src="../images/performance_experiment.png" alt="performance_experiment()" width="80%">

Además de la función *performance_experiment()*, se ha desarrollado una función adicional denominada *performance_experiment_mlflow()*, que reproduce la misma lógica experimental, pero incorpora el registro de resultados en formato MLflow. Esta funcionalidad permite almacenar de manera estructurada métricas, parámetros y artefactos, facilitando la integración con un servidor MLflow para la gestión y trazabilidad de experimentos. Su inclusión responde a la adopción de MLflow como estándar de facto en la experimentación en Ciencia de Datos, con el objetivo de que cualquier persona que utilice este repositorio y quiera ampliar la experimentación pueda utilizarlo si lo desea.

In [7]:
# Selecciona la funcion de acuerdo al parámetro USE_MLFLOW configurado al inicio del notebook
if USE_MLFLOW:
    experiment_method = performance_experiment_mlflow
else:
    experiment_method = performance_experiment

A continuación se ejecutan los experimentos para cada uno de los modelos descritos

In [8]:
################################
# Multinomial Logistic Regression
################################

LR_pipeline = Pipeline([
    ("model", LogisticRegression(
        class_weight="balanced",
        solver="lbfgs",
        penalty="l2",
        random_state=SEED
    ))
])

LR_param_grid = {
    "model__C": [10, 100, 1000, 10000],
}



LR_global_results_df, LR_cm_df, LR_class_report_df = experiment_method(
    config=cfg,
    pipeline=LR_pipeline,
    param_grid=LR_param_grid,
    experiment_name="Logistic_Regression"
)


NESTED CROSS-VALIDATION EXPERIMENT
Experiment Name      : Logistic_Regression
Outer CV Folds       : 5
Inner CV Folds       : 3

Hyperparameter Grid:
model__C                 : [10, 100, 1000, 10000]

--------------------------------------------------------------------------------
OUTER FOLD [1/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (3 folds)
Starting GridSearchCV (metric='f1_macro') ...
Fitting 3 folds for each of 4 candidates, totalling 12 fits

--------------------------------------------------------------------------------
OUTER FOLD [2/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (3 folds)
Starting GridSearchCV (metric='f1_macro') ...
Fitting 3 folds for each of 4 candidates, totalling 12 fits

------------------------------------------------

In [9]:
################################
# Decision Tree
################################

DT_pipeline = Pipeline([
    ("model", DecisionTreeClassifier(
        class_weight = "balanced",
        max_depth = 6,
        random_state=SEED
    ))
])

DT_param_grid = {
    "model__max_leaf_nodes": [10, 15, 20, 25, 30],
    "model__min_samples_leaf": [5, 10, 15],
    "model__criterion": ["gini", "entropy"],
}



DT_global_results_df, DT_cm_df, DT_class_report_df = experiment_method(
    config=cfg,
    pipeline=DT_pipeline,
    param_grid=DT_param_grid,
    experiment_name="Decision_Tree"
)


NESTED CROSS-VALIDATION EXPERIMENT
Experiment Name      : Decision_Tree
Outer CV Folds       : 5
Inner CV Folds       : 3

Hyperparameter Grid:
model__max_leaf_nodes    : [10, 15, 20, 25, 30]
model__min_samples_leaf  : [5, 10, 15]
model__criterion         : ['gini', 'entropy']

--------------------------------------------------------------------------------
OUTER FOLD [1/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (3 folds)
Starting GridSearchCV (metric='f1_macro') ...
Fitting 3 folds for each of 30 candidates, totalling 90 fits

--------------------------------------------------------------------------------
OUTER FOLD [2/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (3 folds)
Starting GridSearchCV (metric='f1_macro') ...
Fitting 3 folds for each of 30

In [10]:
################################
# SVM With RBF Kernel
################################

SVM_pipeline = Pipeline([
    ("model", SVC(
        kernel = "rbf",
        decision_function_shape = 'ovr',
        class_weight = "balanced",
        random_state =SEED
    ))
])

SVM_param_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__gamma": ["scale", "auto", 0.01, 0.1]
}



SVM_global_results_df, SVM_cm_df, SVM_class_report_df = experiment_method(
    config=cfg,
    pipeline=SVM_pipeline,
    param_grid=SVM_param_grid,
    experiment_name="SVM"
)


NESTED CROSS-VALIDATION EXPERIMENT
Experiment Name      : SVM
Outer CV Folds       : 5
Inner CV Folds       : 3

Hyperparameter Grid:
model__C                 : [0.1, 1, 10, 100]
model__gamma             : ['scale', 'auto', 0.01, 0.1]

--------------------------------------------------------------------------------
OUTER FOLD [1/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (3 folds)
Starting GridSearchCV (metric='f1_macro') ...
Fitting 3 folds for each of 16 candidates, totalling 48 fits

--------------------------------------------------------------------------------
OUTER FOLD [2/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (3 folds)
Starting GridSearchCV (metric='f1_macro') ...
Fitting 3 folds for each of 16 candidates, totalling 48 fits

-----------

In [11]:
################################
# Random Forest
################################

RF_pipeline = Pipeline([
    ("model", RandomForestClassifier(
        class_weight = "balanced_subsample",
        random_state=SEED
    ))
])

RF_param_grid = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [5, 10, None],
    "model__max_features": ["sqrt", 0.3, 0.4],
    "model__criterion": ['gini', 'entropy'],
    "model__min_samples_split": [2, 5],
}



RF_global_results_df, RF_cm_df, RF_class_report_df = experiment_method(
    config=cfg,
    pipeline=RF_pipeline,
    param_grid=RF_param_grid,
    experiment_name="Random_Forest"
)


NESTED CROSS-VALIDATION EXPERIMENT
Experiment Name      : Random_Forest
Outer CV Folds       : 5
Inner CV Folds       : 3

Hyperparameter Grid:
model__n_estimators      : [100, 200, 300]
model__max_depth         : [5, 10, None]
model__max_features      : ['sqrt', 0.3, 0.4]
model__criterion         : ['gini', 'entropy']
model__min_samples_split : [2, 5]

--------------------------------------------------------------------------------
OUTER FOLD [1/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (3 folds)
Starting GridSearchCV (metric='f1_macro') ...
Fitting 3 folds for each of 108 candidates, totalling 324 fits

--------------------------------------------------------------------------------
OUTER FOLD [2/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (3 fold

In [12]:
################################
# XGBoost
################################

XG_pipeline = Pipeline([
    ("model", XGBClassifier(
        random_state = SEED,
        sampling_method = "uniform",
        objective= "multi:softmax",
        eval_metric="mlogloss",
        ))
])

XG_param_grid = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [5, 10],
    "model__learning_rate": [0.01, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
    "model__gamma": [0, 0.3],
    "model__reg_lambda": [1, 5]
}



XG_global_results_df, XG_cm_df, XG_class_report_df = experiment_method(
    config=cfg,
    pipeline=XG_pipeline,
    param_grid=XG_param_grid,
    experiment_name="XGBoost"
)


NESTED CROSS-VALIDATION EXPERIMENT
Experiment Name      : XGBoost
Outer CV Folds       : 5
Inner CV Folds       : 3

Hyperparameter Grid:
model__n_estimators      : [100, 300]
model__max_depth         : [5, 10]
model__learning_rate     : [0.01, 0.1]
model__subsample         : [0.8, 1.0]
model__colsample_bytree  : [0.8, 1.0]
model__gamma             : [0, 0.3]
model__reg_lambda        : [1, 5]

--------------------------------------------------------------------------------
OUTER FOLD [1/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
Inner CV splits successfully loaded (3 folds)
Starting GridSearchCV (metric='f1_macro') ...
Fitting 3 folds for each of 128 candidates, totalling 384 fits

--------------------------------------------------------------------------------
OUTER FOLD [2/5]
--------------------------------------------------------------------------------
Train samples:   513 | Test samples:   129
In